***EDA Barrios*** 

Es necesario este dataset para saber el barrio y comuna de cada kiosco, y para poder graficar cada barrio.

En total hay 48 barrios dentro de CABA.

No hay duplicados.

Los valores de las coordenadas son válidas.

Descarto el campo `_rescued_data`, ya que vino totalmente nulo.

No hay valores raros, como "


In [0]:
%sql

describe proyecto_final.raw.barrios_bronze

In [0]:
%sql
select * from proyecto_final.raw.barrios_bronze

In [0]:
%sql
-- Cantidad de Nulos
select count(*) as total_registros,
count(*) - count(id) as id_nulos,
count(*) - count(objeto) as objeto_nulos,
count(*) - count(nombre) as nombre_nulos,
count(*) - count(comuna) as comuna_nulos,
count(*) - count(perimetro_) as perimetro_nulos,
count(*) - count(area_metro) as area_metro_nulos,
count(*) - count(geometry) as geometry_nulos,
count(*) - count(_rescued_data) as _rescued_data_nulos
from proyecto_final.raw.barrios_bronze

-- No hay campos con nulos

In [0]:
%sql
--Duplicados
select 
id,
nombre as nombre_barrio,
count(*) cantidad
from proyecto_final.raw.barrios_bronze
group by id, nombre
having count(*) >1

In [0]:
%sql

--Barrios con mayor area numerados
select 
row_number() over (order by area_metro desc) as pos,
id,
nombre as nombre_barrio,
area_metro
from proyecto_final.raw.barrios_bronze
limit 5;



In [0]:
%sql

-- Busco si en los puntos de geometry hay alguno invalido
SELECT 
    nombre,
    objeto,
    geometry,
    st_isvalid(st_geomfromwkt(geometry)) as es_valida
FROM proyecto_final.raw.barrios_bronze
WHERE st_isvalid(st_geomfromwkt(geometry)) = false;

In [0]:
%sql

CREATE OR REPLACE VIEW v_barrios_limpieza AS
SELECT 
  id AS barrio_id,
  TRIM(nombre) AS barrio_nombre, -- Mantengo el nombre 
  LOWER(TRIM(objeto)) AS tipo_entidad,
  comuna AS comuna_id,
  ROUND(perimetro_, 2) AS perimetro_m,
  ROUND(area_metro, 2) AS area_m2,
  geometry AS coordenadas
FROM proyecto_final.raw.barrios_bronze
WHERE nombre IS NOT NULL 
  AND geometry IS NOT NULL;

In [0]:
select * from v_barrios_limpieza